# Triangle random intersection graphs: run the experiment

How does the size of the 3-core change as individuals join more
communities? We explore this question by generating random graphs and
measuring the fraction of individuals left after 3-core pruning.
The first two notebooks provide the graph representation and
algorithms; here we choose the settings, repeat the experiment, and
save every measurement for later analysis.

Select the project's `.venv` kernel, review **Experiment settings**
below, then run the cells from top to bottom. **Run All repeats the
experiment and replaces the selected CSV file.** Paths are measured
from the project root. Once the simulation is complete, open
`04_triangle_analysis.ipynb` to plot the saved measurements.

## What we simulate and measure

Each of the $n$ individuals receives a number of community memberships,
sampled as $D_i \sim \operatorname{Poisson}(\lambda)$. Each community
has three slots, so the total number of memberships must be divisible
by three. If it is not, we discard **all** the sampled counts and draw
a new vector. This samples the Poisson counts conditional on a
divisible total, rather than adjusting one individual's count.
We then randomly match the memberships to community slots.
Different individuals who share a community are connected. Membership
in the same community more than once creates no self-edge, and a pair
of individuals is connected by at most one edge.

The parameter $\lambda$ is the mean of the Poisson membership
distribution before conditioning. It does not directly specify the
number of graph neighbours: each membership can add two neighbours.
For large sparse graphs, the mean graph degree is approximately
$2\lambda$; repeated memberships and shared neighbours affect this
approximation in finite graphs.

For each choice of $(n,\lambda)$, we sample several independent graphs
and measure
$$X_{n,\lambda} = \frac{|\operatorname{Core}_3(G_{n,\lambda})|}{n}.$$
The 3-core is what remains after repeatedly removing individuals with
fewer than three remaining neighbours. Its size is divided by **all**
$n$ individuals, including those with no edges. The algorithm works
on a copy, so the sampled graph is available for other measurements.

In [ ]:
from pathlib import Path

# Find the other notebooks whether we start here or at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

# Load the graph classes and algorithms; this does not run a simulation.
%run "{NOTEBOOK_DIR}/02_algorithms.ipynb"

In [ ]:
import csv
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed

# Interpret output paths from the project root, wherever the kernel starts.
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Experiment settings

The dictionary below contains all the simulation settings. Choose the
graph size with `n` and the number of independent graphs sampled at
each lambda with `repeats`. Larger values give more information but
take longer to run. For a quick first run, reduce both values.

The values from `lambda_min` to `lambda_max` are equally spaced, with
`lambda_count` points including both endpoints. Remember that lambda
controls community memberships, rather than graph degree.

The `seed` makes the random experiment repeatable. Set `jobs=1` to
run one trial at a time, or `jobs=-1` to use all available CPU cores.
Choose a new `data_file` to keep results from an earlier run, and use
that same path in the analysis notebook.

In [ ]:
config = {
    "n": 1000,  # Number of individuals in each graph, including those with no edges.
    "repeats": 20,  # Number of independent graphs to sample at each lambda value.
    "lambda_min": 0.7,  # Smallest Poisson membership parameter to investigate.
    "lambda_max": 2.5,  # Largest Poisson membership parameter to investigate.
    "lambda_count": 41,  # Number of evenly spaced values, including both ends.
    "seed": 2026,  # Starting seed used to create a separate seed for every trial.
    "jobs": 1,  # Use 1 for one trial at a time, or -1 for all CPU cores.
    "data_file": "results/runs/triangle_transition.csv",  # This file is replaced.
}
config

## Running individual trials and making them repeatable

One **trial** means generating one graph and measuring its 3-core.
`triangle_trial` does this using a supplied random seed.
`simulate_triangle` repeats the trial at each lambda value, giving
every trial its own seed before assigning work to CPU workers.
As a result, changing the number of workers does not change the
measurements. Results are saved in lambda order, then by trial number
(`replicate`, counted from zero).

Using a saved trial seed with the same code and dependencies
reproduces the graph's connections and core size. Node identifiers are
random UUIDs generated separately, so their names will differ.
Changing the lambda grid or number of repeats can change which seed
is assigned to a trial; the CSV records the seed actually used.

In [ ]:
CSV_FIELDS = ("model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction")


def triangle_trial(n: int, lam: float, replicate: int, seed: int) -> dict:
    """Generate one graph and save its core size, settings, and random seed."""
    if n <= 0:
        raise ValueError("n must be positive")
    rng = np.random.default_rng(seed)
    graph = construct_special_triangle_RIG(n, lam, rng=rng)
    core = construct_k_core(graph, 3)
    core_size = len(core.nodes)
    return {
        "model": "triangle_rig",
        "n": n,
        "lambda": lam,
        "replicate": replicate,
        "seed": seed,
        "core_size": core_size,
        "core_fraction": core_size / n,
    }


def simulate_triangle(
    n: int, lambdas, repeats: int, seed: int = 2026, jobs: int = 1
) -> list[dict]:
    """Repeat the experiment at each lambda and return one row per graph.

    Results follow the supplied lambda order, then the trial number.
    Use ``jobs=1`` to run one trial at a time or ``jobs=-1`` to use all
    CPU cores. Every trial receives its seed before work is assigned,
    so the number of workers does not affect the measurements.
    """
    grid = [float(lam) for lam in lambdas]
    if n <= 0 or repeats <= 0 or seed < 0 or jobs == 0:
        raise ValueError(
            "Require positive n/repeats, nonnegative seed, and nonzero jobs"
        )
    if (
        not grid
        or len(set(grid)) != len(grid)
        or any(not np.isfinite(lam) or lam < 0 for lam in grid)
    ):
        raise ValueError("Use distinct, finite, nonnegative lambda values")

    # Give every trial its own random stream before starting any workers.
    # Store seeds as integers so the CSV preserves their exact values.
    child_seeds = iter(np.random.SeedSequence(seed).spawn(len(grid) * repeats))
    tasks = []
    for lam in grid:
        for replicate in range(repeats):
            trial_seed = int(next(child_seeds).generate_state(1, dtype=np.uint64)[0])
            tasks.append((n, lam, replicate, trial_seed))

    return Parallel(n_jobs=jobs)(delayed(triangle_trial)(*task) for task in tasks)


def save_results(rows: list[dict], path: str | Path) -> None:
    """Save one row per trial, replacing the CSV and creating folders as needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

## Run the experiment and save the measurements

This cell generates all the graphs, so it takes most of the running
time. It saves each trial separately, allowing us to change plots and
summaries later without repeating the simulation.

Each CSV row records the model name, graph size (`n`), lambda, trial
number (`replicate`), random seed, and the two measurements:
`core_size` is the number of individuals in the 3-core, and
`core_fraction` is that count divided by `n`. The seed is saved as an
integer so it can be reused exactly.

In [ ]:
lambda_axis = np.linspace(
    config["lambda_min"], config["lambda_max"], config["lambda_count"]
)
started = perf_counter()
rows = simulate_triangle(
    n=config["n"],
    lambdas=lambda_axis,
    repeats=config["repeats"],
    seed=config["seed"],
    jobs=config["jobs"],
)
data_path = PROJECT_ROOT / config["data_file"]
save_results(rows, data_path)
print(
    f"Saved {len(rows)} realizations to {data_path} "
    f"in {perf_counter() - started:.1f} seconds."
)

Next, open `04_triangle_analysis.ipynb` to see how the core fraction
changes with lambda and how much it varies between sampled graphs.
To investigate how graph size affects the results, repeat with other
values of `n`. Use a different output filename for each experiment
so the measurements remain available for comparison.